# Semantic Kernel 

اس کوڈ نمونے میں، آپ سادہ ایجنٹ بنانے کے لیے [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI فریم ورک استعمال کریں گے۔

اس نمونے کا مقصد وہ مراحل دکھانا ہے جو بعد میں اضافی کوڈ نمونوں میں مختلف ایجنٹک پیٹرنز کو نافذ کرتے وقت لاگو کیے جائیں گے۔


## ضروری پائتھن پیکجز درآمد کریں


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## کلائنٹ بنانا

اس مثال میں، ہم LLM تک رسائی کے لیے [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) استعمال کریں گے۔

`ai_model_id` کو `gpt-4o-mini` پر سیٹ کیا گیا ہے۔ مختلف نتائج دیکھنے کے لیے ماڈل کو GitHub Models مارکیٹ پلیس میں دستیاب کسی دوسرے ماڈل پر تبدیل کرنے کی کوشش کریں۔

GitHub Models کے لیے درکار `base_url` کے لیے `Azure Inference SDK` استعمال کرنے کے لیے، ہم Semantic Kernel کے اندر `OpenAIChatCompletion` کنیکٹر کا استعمال کریں گے۔ دوسرے [دستیاب کنیکٹرز](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) بھی ہیں جو آپ کو Semantic Kernel کو دوسرے ماڈل فراہم کنندگان کے ساتھ استعمال کرنے کی اجازت دیتے ہیں۔


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## ایجنٹ کی تخلیق

یہاں، ہم `TravelAgent` نامی ایجنٹ بناتے ہیں۔

اس مثال میں، ہم بہت بنیادی ہدایات استعمال کرتے ہیں۔ آپ بلا جھجھک ان ہدایات میں ترمیم کر سکتے ہیں تاکہ دیکھ سکیں کہ ایجنٹ کا برتاؤ کیسے بدلتا ہے۔


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## ایجنٹس کو چلانا

اب ہم ایجنٹ کو چلانے کے لیے `ChatHistory` ترتیب دے کر اور اس کے اندر `system_message` شامل کر کے اجرا کر سکتے ہیں۔ ہم وہ `AGENT_INSTRUCTIONS` استعمال کریں گے جو ہم نے پہلے طے کیے تھے۔

جب یہ سب ترتیب دیے جائیں، تو ہم `user_inputs` بناتے ہیں، جو اس بات کی نمائندگی کرتے ہیں کہ صارف ایجنٹ کو کیا بھیج رہا ہے۔ اس مثال میں، پیغام کو `Plan me a sunny vacation` پر سیٹ کیا گیا ہے۔

آپ اس پیغام میں ترمیم کر سکتے ہیں تاکہ دیکھیں کہ ایجنٹ مختلف طریقے سے کیسے ردعمل دیتا ہے۔


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنبیہ**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجموں میں غلطیاں یا بے ضابطگیاں ہو سکتی ہیں۔ اصل دستاویز اپنی اصل زبان میں ہی معتبر ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم پر عائد نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
